# Modeling Pipeline

## Overview

**Objective**  
Build a robust and reproducible modeling pipeline for customer churn prediction using engineered features from the feature engineering step.

**Key principles**
- Use a consistent feature set (no re-engineering in modeling)
- Avoid data leakage
- Use cross-validation for reliable evaluation
- Improve stability using multi-seed training
- Combine models using ensemble for better performance

**Models used**
- LightGBM (primary model)
- XGBoost (secondary model)

**Final output**
- Out-of-fold (OOF) predictions for validation
- Test predictions for submission

## 1. Load Data

Load processed datasets from the feature engineering step.

These datasets already contain:
- Selected features
- One-hot encoded variables
- Cleaned and aligned train/test columns

Important:
- No additional preprocessing or encoding is applied here
- This ensures consistency between training and inference

In [3]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/processed/"

train = pd.read_csv(DATA_PATH + "train_final.csv")
test = pd.read_csv(DATA_PATH + "test_final.csv")

TARGET = "Churn"

X = train.drop(columns=[TARGET])
y = train[TARGET]
X_test = test.copy()

print("Train:", X.shape, "Test:", X_test.shape)

Train: (594194, 25) Test: (254655, 25)


## 2. Cross Validation Strategy

Stratified K-Fold is used to:
- Preserve churn distribution across folds
- Avoid biased evaluation due to class imbalance

Why this matters:
- Churn dataset is imbalanced
- Random split may produce misleading results

Output:
- Out-of-fold (OOF) predictions  
  These simulate predictions on unseen data and are used for reliable AUC evaluation.


In [4]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

SEED = 42
N_SPLITS = 5

def run_lgb_cv(X, y, seed=42):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(
            n_estimators=3000,
            learning_rate=0.02,
            num_leaves=48,
            min_child_samples=50,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.5,
            reg_lambda=0.5,
            random_state=seed,
            n_jobs=-1
        )

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
        )

        oof[val_idx] = model.predict_proba(X_val)[:, 1]

    return roc_auc_score(y, oof)

## 3. Multi-Seed Training

Instead of using a single random seed, the model is trained with multiple seeds.

Why:
- Different seeds produce slightly different data splits
- Reduces randomness and variance in performance
- Leads to more stable predictions

Final predictions are averaged across all seeds.

## LightGBM Model

LightGBM is used as the primary model because:
- It performs well on tabular data
- It is efficient and fast
- It captures non-linear relationships

Key design choices:
- Lower learning rate for better generalization
- Moderate number of leaves to reduce overfitting
- Regularization (L1/L2) to stabilize training

Early stopping:
- Stops training when validation performance no longer improves
- Helps prevent overfitting

In [5]:
SEEDS = [42, 52, 62]

oof_lgb = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))

for seed in SEEDS:
    print(f"\nLGB SEED {seed}")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof_seed = np.zeros(len(X))
    test_seed = np.zeros(len(X_test))

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(
            n_estimators=3000,
            learning_rate=0.02,
            num_leaves=48,
            min_child_samples=50,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.5,
            reg_lambda=0.5,
            random_state=seed,
            n_jobs=-1
        )

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)]
        )

        oof_seed[val_idx] = model.predict_proba(X_val)[:, 1]
        test_seed += model.predict_proba(X_test)[:, 1] / N_SPLITS

    oof_lgb += oof_seed / len(SEEDS)
    test_lgb += test_seed / len(SEEDS)

print("LGB CV:", roc_auc_score(y, oof_lgb))


LGB SEED 42
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016441 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 631
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 25
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1723]	valid_0's binary_logloss: 0.297968
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 107054, nu

## 4. XGBoost Model

XGBoost is used as a secondary model.

Why:
- Uses a different tree construction approach than LightGBM
- Learns slightly different patterns
- Improves ensemble performance

Strategy:
- Similar structure to LightGBM
- Slightly stronger regularization to avoid overfitting


In [6]:
import xgboost as xgb

oof_xgb = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))

for seed in SEEDS:
    print(f"\nXGB SEED {seed}")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof_seed = np.zeros(len(X))
    test_seed = np.zeros(len(X_test))

    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = xgb.XGBClassifier(
            n_estimators=3000,
            learning_rate=0.02,
            max_depth=4,
            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.8,
            gamma=1,
            reg_alpha=0.5,
            reg_lambda=1.0,
            eval_metric="auc",
            random_state=seed,
            n_jobs=-1,
            tree_method="hist"
        )

        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        oof_seed[val_idx] = model.predict_proba(X_val)[:, 1]
        test_seed += model.predict_proba(X_test)[:, 1] / N_SPLITS

    oof_xgb += oof_seed / len(SEEDS)
    test_xgb += test_seed / len(SEEDS)

print("XGB CV:", roc_auc_score(y, oof_xgb))


XGB SEED 42

XGB SEED 52

XGB SEED 62
XGB CV: 0.9166459321331437



## 5. Ensemble

Predictions from LightGBM and XGBoost are combined.

Why ensemble:
- Different models capture different patterns
- Averaging reduces model-specific errors
- Typically improves performance

Method:
- Weighted average of predictions
- Optimal weight is found using OOF predictions

Formula:
```
final_prediction = w * LGB + (1 - w) * XGB
```

The best weight is selected based on validation AUC.

In [7]:
best_score = 0
best_w = 0

for w in np.arange(0.0, 1.01, 0.01):
    oof_comb = w * oof_lgb + (1 - w) * oof_xgb
    score = roc_auc_score(y, oof_comb)

    if score > best_score:
        best_score = score
        best_w = w

print(f"Best LGB weight: {best_w:.2f}")
print(f"Best Ensemble AUC: {best_score:.5f}")

Best LGB weight: 0.50
Best Ensemble AUC: 0.91676


## 7. Final Prediction

Apply the best ensemble weights to test predictions.

Important:
- Use probability outputs, not binary predictions
- Kaggle evaluates using AUC, which requires probabilities

In [8]:
test_preds = best_w * test_lgb + (1 - best_w) * test_xgb